In [1]:
from pathlib import Path
import importlib.metadata
import json
import torch

project = Path("/content/drive/MyDrive/FacialVisualProfile")
day19 = project / "runs/benchmark/day19"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for package in (
    "torch", "torchvision", "facenet-pytorch",
    "numpy", "Pillow", "scikit-learn",
):
    try:
        print(f"{package}: {importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

for relative in (
    "day19_benchmark_report.json",
    "execution_binding.json",
    "test_jpeg_sha256.json",
    "source/scripts/day19_evaluate_benchmark.py",
    "source/frozen_inputs/day19_handoff_manifest.json",
):
    print(relative, "exists:", (day19 / relative).is_file())

report_path = day19 / "day19_benchmark_report.json"
if report_path.is_file():
    report = json.loads(report_path.read_bytes())
    assert report["day19_scope_complete"] is True
    assert report["condition_count"] == 9
    assert report["test_sample_count"] == 19962
    assert report["threshold_fitting_performed"] is False
    print("Completed Day19 report: PASS")

print(
    "Local test images available:",
    Path("/content/day19_benchmark_data/raw/img_align_celeba").is_dir(),
)
print("Day20 preparation inspection: COMPLETE")
print("No new test inference performed")

CUDA available: True
GPU: Tesla T4
torch: 2.11.0+cu130
torchvision: 0.26.0+cu130
facenet-pytorch: NOT INSTALLED
numpy: 2.1.3
Pillow: 11.3.0
scikit-learn: 1.6.1
day19_benchmark_report.json exists: False
execution_binding.json exists: False
test_jpeg_sha256.json exists: False
source/scripts/day19_evaluate_benchmark.py exists: False
source/frozen_inputs/day19_handoff_manifest.json exists: False
Local test images available: False
Day20 preparation inspection: COMPLETE
No new test inference performed


In [2]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

project = Path("/content/drive/MyDrive/FacialVisualProfile")
day19 = project / "runs/benchmark/day19"

print("Project exists:", project.is_dir())

for relative in (
    "day19_benchmark_report.json",
    "execution_binding.json",
    "test_jpeg_sha256.json",
    "source/scripts/day19_evaluate_benchmark.py",
    "source/frozen_inputs/day19_handoff_manifest.json",
):
    print(relative, "exists:", (day19 / relative).is_file())

if project.is_dir():
    benchmark = project / "runs/benchmark"
    print("\nBenchmark directories:")
    if benchmark.is_dir():
        for path in sorted(benchmark.iterdir()):
            print(path.name)

Mounted at /content/drive
Project exists: True
day19_benchmark_report.json exists: True
execution_binding.json exists: True
test_jpeg_sha256.json exists: True
source/scripts/day19_evaluate_benchmark.py exists: True
source/frozen_inputs/day19_handoff_manifest.json exists: True

Benchmark directories:
day19


In [3]:
from google.colab import files
from pathlib import Path
import hashlib
import importlib.util

uploaded = files.upload()
assert len(uploaded) == 1, "请只上传 day20_prepare.py"
data = next(iter(uploaded.values()))

assert hashlib.sha256(data).hexdigest() == (
    "9431c97231bded7aa14fd51a13727004182"
    "cf3053fae08ee3047f77d4c51ba3f"
), "脚本内容不匹配"

path = Path("/content/day20_prepare.py")
path.write_bytes(data)

spec = importlib.util.spec_from_file_location("day20_prepare", path)
preparation = importlib.util.module_from_spec(spec)
spec.loader.exec_module(preparation)

DAY20_CONTEXT = preparation.prepare_day20()

Saving day20_prepare.py to day20_prepare.py
Original execution environment: MATCH
Restoring r18_seed0: verifying snapshot and checkpoint...
r18_seed0: strict restoration PASS
Restoring r50_seed0: verifying snapshot and checkpoint...
r50_seed0: strict restoration PASS
Restoring m3_seed0: verifying snapshot and checkpoint...
m3_seed0: strict restoration PASS
Original detector weights: MATCH
Restoring local image ZIP from Drive...
Copied 256 MiB
Copied 512 MiB
Copied 768 MiB
Copied 1024 MiB
Copied 1280 MiB
Restored test JPEGs: 2000/19962
Restored test JPEGs: 4000/19962
Restored test JPEGs: 6000/19962
Restored test JPEGs: 8000/19962
Restored test JPEGs: 10000/19962
Restored test JPEGs: 12000/19962
Restored test JPEGs: 14000/19962
Restored test JPEGs: 16000/19962
Restored test JPEGs: 18000/19962
Restored test JPEGs: 19962/19962
Original 19962 test JPEG hashes: MATCH
Day20 environment, models and test-data preparation: PASS
No model inference or new test metrics performed


In [4]:
import csv
import hashlib
import importlib
import importlib.metadata
import importlib.util
import json
import os
from pathlib import Path
import platform
import random
import subprocess
import sys
import types
import zipfile

import numpy as np
import torch


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def restore_file(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        assert path.read_bytes() == data, f"已有文件内容不同，已保留：{path}"
    else:
        temporary = path.with_suffix(path.suffix + ".restoring")
        temporary.write_bytes(data)
        temporary.replace(path)


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
DAY19 = PROJECT / "runs/benchmark/day19"
ORIGINAL = json.loads((DAY19 / "execution_binding.json").read_bytes())

# 只安装原版本 detector package，保持现有 torch 等依赖。
version = ORIGINAL["environment"]["facenet-pytorch"]
try:
    installed = importlib.metadata.version("facenet-pytorch")
except importlib.metadata.PackageNotFoundError:
    installed = None

if installed != version:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--no-deps", "facenet-pytorch==" + version,
    ])
    importlib.invalidate_caches()

assert torch.cuda.is_available(), "请使用 GPU 运行时"

current = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "gpu": torch.cuda.get_device_name(0),
    "cuda_runtime": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "environment": {
        name: importlib.metadata.version(name)
        for name in ORIGINAL["environment"]
    },
}

differences = []
for name, value in current.items():
    if name == "environment":
        differences.extend(
            (package, ORIGINAL[name][package], now)
            for package, now in value.items()
            if ORIGINAL[name][package] != now
        )
    elif ORIGINAL[name] != value:
        differences.append((name, ORIGINAL[name], value))

if differences:
    for name, before, now in differences:
        print(f"{name}: Day19={before!r}; current={now!r}")
    raise RuntimeError("环境与 Day19 不同，请贴出以上差异后再继续")

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.set_num_threads(ORIGINAL["torch_threads"])
torch.backends.cudnn.deterministic = ORIGINAL["cudnn_deterministic"]
torch.backends.cudnn.benchmark = ORIGINAL["cudnn_benchmark"]
torch.backends.cuda.matmul.allow_tf32 = ORIGINAL["tf32"]
torch.backends.cudnn.allow_tf32 = ORIGINAL["tf32"]

WORKSPACE = Path("/content/day19_benchmark_workspace")
saved_sources = DAY19 / "source/frozen_inputs"
handoff = json.loads(
    (saved_sources / "day19_handoff_manifest.json").read_bytes()
)

for name, expected in handoff["files"].items():
    relative = Path(name)
    assert not relative.is_absolute() and ".." not in relative.parts
    source = saved_sources / relative
    assert sha_file(source) == expected, f"冻结源码发生变化：{name}"
    restore_file(WORKSPACE / relative, source.read_bytes())

config_path = WORKSPACE / "configs/benchmark_v1.json"
assert sha_file(config_path) == ORIGINAL["config_sha256"]
CONFIG = json.loads(config_path.read_bytes())

freeze_path = (
    WORKSPACE / "docs/results/benchmark_freeze/day17/freeze_audit.json"
)
assert sha_file(freeze_path) == ORIGINAL["freeze_audit_sha256"]
freeze = json.loads(freeze_path.read_bytes())
assert freeze["passed"] is True

for name, expected in freeze["source_and_artifact_sha256"].items():
    assert sha_file(WORKSPACE / name) == expected, name

manifest_path = WORKSPACE / CONFIG["official_test_manifest"]
assert sha_file(manifest_path) == ORIGINAL["manifest_sha256"]
with manifest_path.open(newline="") as stream:
    TEST_NAMES = [row["sample_id"] for row in csv.DictReader(stream)]
assert len(TEST_NAMES) == len(set(TEST_NAMES)) == 19962

print("Day19 environment and frozen inputs: MATCH")

Day19 environment and frozen inputs: MATCH


In [5]:
MODELS = {}
MODEL_PROVENANCE = {}

model_specs = {
    "r18_seed0": ("models/resnet18.py", "ResNet18Attributes"),
    "r50_seed0": ("models/resnet50.py", "ResNet50Attributes"),
    "m3_seed0": ("models/resnet18_geometry.py", "ResNet18GeometryAttributes"),
}

for run, (relative, class_name) in model_specs.items():
    print(f"Restoring {run}...", flush=True)
    run_root = PROJECT / "runs" / run
    training = json.loads((run_root / "config.json").read_bytes())
    assert training["attribute_names"] == CONFIG["attribute_names"]

    for name, expected in training["source_sha256"].items():
        assert sha_file(run_root / "source" / name) == expected, name

    selected = CONFIG["threshold_bindings"][run]
    checkpoint_path = run_root / selected["checkpoint"]
    assert sha_file(checkpoint_path) == selected["checkpoint_sha256"]

    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=False
    )
    expected_epoch = int(
        Path(selected["checkpoint"]).stem.split("_")[-1]
    )
    assert checkpoint["epoch"] == expected_epoch
    assert checkpoint["config"]["source_sha256"] == training["source_sha256"]
    assert checkpoint["config"]["attribute_names"] == CONFIG["attribute_names"]

    source = run_root / "source" / relative
    module = load_module("day20_" + run, source)
    model = getattr(module, class_name)(
        num_attributes=24, pretrained=False
    )
    state_keys = [
        key for key in ("model_state", "model_state_dict")
        if key in checkpoint
    ]
    assert len(state_keys) == 1
    model.load_state_dict(checkpoint[state_keys[0]], strict=True)
    MODELS[run] = model.to("cuda").eval()

    MODEL_PROVENANCE[run] = {
        "checkpoint_sha256": selected["checkpoint_sha256"],
        "model_source_sha256": sha_file(source),
        "state_key": state_keys[0],
    }
    assert MODEL_PROVENANCE[run] == ORIGINAL["models"][run]
    del checkpoint
    print(f"{run}: strict loading PASS", flush=True)

# 用独立命名空间加载冻结 preprocessing，避免 datasets 包冲突。
package_name = "day20_frozen_datasets"
package = types.ModuleType(package_name)
package.__path__ = [str(WORKSPACE / "datasets")]
sys.modules[package_name] = package

preprocessing = importlib.import_module(
    package_name + ".benchmark_preprocessing"
)
corruptions = importlib.import_module(package_name + ".corruptions")

detector_module = load_module(
    "day20_mtcnn", WORKSPACE / "detectors/mtcnn.py"
)
DETECTOR = detector_module.MTCNNDetector(device="cpu")

digest = hashlib.sha256()
for name, tensor in sorted(DETECTOR.model.state_dict().items()):
    value = tensor.detach().cpu().contiguous().numpy()
    digest.update(name.encode("utf-8"))
    digest.update(str(value.dtype).encode("ascii"))
    digest.update(str(value.shape).encode("ascii"))
    digest.update(value.tobytes())

DETECTOR_WEIGHTS_SHA256 = digest.hexdigest()
assert DETECTOR_WEIGHTS_SHA256 == ORIGINAL["detector_weights_sha256"]

quality = json.loads((
    WORKSPACE
    / "docs/results/corruption_preparation/day16/detector_quality_report.json"
).read_bytes())
assert DETECTOR.CONFIG == quality["detector_config"]
assert list(DETECTOR.POINT_ORDER) == quality["point_order"]
assert DETECTOR_WEIGHTS_SHA256 == quality["detector_weights_sha256"]

print("Three models and detector restoration: PASS")
print("No model inference performed")

Restoring r18_seed0...
r18_seed0: strict loading PASS
Restoring r50_seed0...
r50_seed0: strict loading PASS
Restoring m3_seed0...
m3_seed0: strict loading PASS
Three models and detector restoration: PASS
No model inference performed


In [6]:
DATA_ROOT = Path("/content/day19_benchmark_data/raw")
image_root = DATA_ROOT / "img_align_celeba"
image_root.mkdir(parents=True, exist_ok=True)

directories = (
    PROJECT / "data",
    PROJECT / "data/raw",
    PROJECT / "data/celeba",
    PROJECT,
)

ANNOTATION_SHA256 = ORIGINAL["annotation_sha256"]
IMAGE_ZIP_SHA256 = ORIGINAL["image_zip_sha256"]

for name, expected in ANNOTATION_SHA256.items():
    destination = DATA_ROOT / name
    if destination.exists():
        assert sha_file(destination) == expected, name
        continue

    matches = [
        directory / name for directory in directories
        if (directory / name).is_file()
        and sha_file(directory / name) == expected
    ]
    assert matches, f"找不到原始标注：{name}"
    restore_file(destination, matches[0].read_bytes())

jpeg_manifest = DAY19 / "test_jpeg_sha256.json"
assert sha_file(jpeg_manifest) == ORIGINAL["test_jpeg_manifest_sha256"]
jpeg_hashes = json.loads(jpeg_manifest.read_bytes())
assert set(jpeg_hashes) == set(TEST_NAMES)

missing = [
    name for name in TEST_NAMES
    if not (image_root / name).is_file()
]

if missing:
    local_zip = Path("/content/day19_img_align_celeba.zip")
    if local_zip.exists():
        assert sha_file(local_zip) == IMAGE_ZIP_SHA256
    else:
        candidates = [
            directory / "img_align_celeba.zip"
            for directory in directories
            if (directory / "img_align_celeba.zip").is_file()
        ]
        assert len(candidates) == 1, f"图片 ZIP 路径不唯一：{candidates}"

        print("Copying original image ZIP from Drive...", flush=True)
        temporary = local_zip.with_suffix(".copying")
        digest = hashlib.sha256()
        copied = 0
        next_progress = 256 * 1024 * 1024

        with candidates[0].open("rb") as source, temporary.open("wb") as target:
            for chunk in iter(lambda: source.read(8 * 1024 * 1024), b""):
                digest.update(chunk)
                target.write(chunk)
                copied += len(chunk)
                if copied >= next_progress:
                    print(f"Copied {copied / 1024**2:.0f} MiB", flush=True)
                    next_progress += 256 * 1024 * 1024

        assert digest.hexdigest() == IMAGE_ZIP_SHA256
        temporary.replace(local_zip)

    wanted = set(missing)
    with zipfile.ZipFile(local_zip) as archive:
        members = {}
        for member in archive.infolist():
            name = member.filename.rsplit("/", 1)[-1]
            if not member.is_dir() and name in wanted:
                assert name not in members, name
                members[name] = member.filename

        assert set(members) == wanted
        for index, name in enumerate(missing, 1):
            data = archive.read(members[name])
            assert hashlib.sha256(data).hexdigest() == jpeg_hashes[name]
            restore_file(image_root / name, data)
            if index % 2000 == 0 or index == len(missing):
                print(f"Restored {index}/{len(missing)} test images", flush=True)

for name in TEST_NAMES:
    assert sha_file(image_root / name) == jpeg_hashes[name], name
assert {
    path.name for path in image_root.iterdir() if path.is_file()
} == set(TEST_NAMES)

# 复用 M3 训练时保存的 Dataset 实现。
dataset_source = PROJECT / "runs/m3_seed0/source/datasets/celeba.py"
assert sha_file(dataset_source) == ORIGINAL["dataset_source_sha256"]

aliases = ("datasets", "datasets.preprocessing")
previous = {name: sys.modules.get(name) for name in aliases}
try:
    sys.modules["datasets"] = package
    sys.modules["datasets.preprocessing"] = importlib.import_module(
        package_name + ".preprocessing"
    )
    dataset_module = load_module("day20_celeba", dataset_source)
finally:
    for name, before in previous.items():
        if before is None:
            sys.modules.pop(name, None)
        else:
            sys.modules[name] = before

previous_directory = Path.cwd()
try:
    os.chdir(WORKSPACE)
    TEST_DATASET = dataset_module.CelebAAttributes(
        split="test", root=DATA_ROOT
    )
finally:
    os.chdir(previous_directory)

assert TEST_DATASET.names == TEST_NAMES
assert TEST_DATASET.attribute_names == CONFIG["attribute_names"]

DAY20_CONTEXT = dict(
    workspace=WORKSPACE,
    project=PROJECT,
    dataset=TEST_DATASET,
    models=MODELS,
    detector=DETECTOR,
    preprocessing=preprocessing,
    corruptions=corruptions,
    model_provenance=MODEL_PROVENANCE,
    detector_weights_sha256=DETECTOR_WEIGHTS_SHA256,
    annotation_sha256=ANNOTATION_SHA256,
    image_zip_sha256=IMAGE_ZIP_SHA256,
)

print("Original 19962 test JPEG hashes: MATCH")
print("Day20 environment, models and test-data preparation: PASS")
print("No model inference or new test metrics performed")

Original 19962 test JPEG hashes: MATCH
Day20 environment, models and test-data preparation: PASS
No model inference or new test metrics performed


In [7]:
# 1. 验证已归档的 Day19 结果，取出 clean 对照。
day19_report_path = DAY19 / "day19_benchmark_report.json"
assert sha_file(day19_report_path) == (
    "79e36c3725d10c2319206023f2b859854"
    "108fc7f1fc176e6998f0d96331b3e1b"
)
day19_report = json.loads(day19_report_path.read_bytes())

assert day19_report["day19_scope_complete"] is True
assert day19_report["condition_count"] == 9
assert day19_report["test_sample_count"] == 19962
assert day19_report["threshold_fitting_performed"] is False
assert day19_report["runtime_errors"] == 0
assert len(day19_report["shard_sha256"]) == 702
assert day19_report["execution_binding_sha256"] == sha_file(
    DAY19 / "execution_binding.json"
)

clean_matches = [
    item for item in day19_report["conditions"]
    if item["condition"]["condition"] == "clean"
    and item["condition"]["severity"] == 0
]
assert len(clean_matches) == 1
clean_reference = clean_matches[0]

# 2. 读取原始运行代码，保留其分批保存、校验、指标和失败处理。
original_runner_path = (
    DAY19 / "source/scripts/day19_evaluate_benchmark.py"
)
assert sha_file(original_runner_path) == ORIGINAL["runner_sha256"]
source = original_runner_path.read_text(encoding="utf-8")

# 以下是 Day20 相对于 Day19 的明确改动。
changes = [
    ("def run_day19(", "def run_day20("),

    (
        'output = project / "runs/benchmark/day19"',
        'output = project / "runs/benchmark/day20"',
    ),

    (
        'if item["condition"] in ("clean", "blur", "brightness")',
        'if item["condition"] in ("rotation", "occlusion", "jpeg")',
    ),

    (
        '[("clean", 0)] + [(kind, severity) for kind in ("blur", "brightness")',
        '[(kind, severity) for kind in ("rotation", "occlusion", "jpeg")',
    ),

    (
        "9 conditions, 19962 images, 3 models",
        "12 conditions, 19962 images, 3 models",
    ),

    # 输入总数和进度估计根据条件数计算。
    ("len(names) * 9", "len(names) * len(conditions)"),

    # 性能下降相对于 Day19 已完成的 clean 结果计算。
    (
        'clean = summaries[0]["models"][run][rule]',
        'clean = clean_reference["models"][run][rule]',
    ),

    (
        "Day19 clean, blur, brightness; remaining conditions and Oracle pending",
        "Day20 rotation, occlusion, jpeg; Oracle pending",
    ),

    (
        '"day19_scope_complete": True',
        '"day20_scope_complete": True, '
        '"r18_r50_m3_condition_count_completed": 21',
    ),

    ("day19_benchmark_report.json", "day20_benchmark_report.json"),
    (
        "Day19 formal test evaluation: PASS",
        "Day20 formal test evaluation: PASS",
    ),
]

for old, new in changes:
    assert old in source, f"原代码与预期不同：{old}"
    source = source.replace(old, new)

# 3. 在写入 Day20 execution binding 前，
#    强制验证与 Day19 共用的环境、模型、数据和规则完全一致。
old = '''    binding_bytes = encoded(binding)
    write_once(output / "execution_binding.json", binding_bytes)'''

new = '''    day19_output = project / "runs/benchmark/day19"
    original_binding_path = day19_output / "execution_binding.json"
    original_binding = json.loads(original_binding_path.read_bytes())
    report_path = day19_output / "day19_benchmark_report.json"
    assert sha_file(report_path) == (
        "79e36c3725d10c2319206023f2b859854"
        "108fc7f1fc176e6998f0d96331b3e1b"
    )
    previous_report = json.loads(report_path.read_bytes())
    assert previous_report["execution_binding_sha256"] == sha_file(
        original_binding_path
    )
    assert previous_report["day19_scope_complete"] is True
    assert previous_report["threshold_fitting_performed"] is False
    assert previous_report["runtime_errors"] == 0

    assert set(binding) == set(original_binding)
    for key in binding:
        if key not in ("runner_sha256", "conditions"):
            assert binding[key] == original_binding[key], (
                f"Shared Day19/Day20 execution setting differs: {key}"
            )

    clean_candidates = [
        item for item in previous_report["conditions"]
        if item["condition"]["condition"] == "clean"
        and item["condition"]["severity"] == 0
    ]
    assert len(clean_candidates) == 1
    clean_reference = clean_candidates[0]

    binding["day19_execution_binding_sha256"] = sha_file(
        original_binding_path
    )
    binding["day19_report_sha256"] = sha_file(report_path)
    binding_bytes = encoded(binding)
    write_once(output / "execution_binding.json", binding_bytes)'''

assert source.count(old) == 1
source = source.replace(old, new)

# 更新文件开头的说明。
end_of_docstring = source.index('"""', 3) + 3
source = (
    '"""Day20: rotation/occlusion/JPEG test evaluation; '
    'reuse verified Day19 implementation and clean reference."""'
    + source[end_of_docstring:]
)

compile(source, "day20_evaluate_benchmark.py", "exec")

# 本地与 Drive 均保存实际运行源码，供归档和断点续跑校验。
DAY20_RUNNER_PATH = Path("/content/day20_evaluate_benchmark.py")
restore_file(DAY20_RUNNER_PATH, source.encode("utf-8"))
restore_file(
    PROJECT / "runs/benchmark/day20/source/scripts/day20_evaluate_benchmark.py",
    source.encode("utf-8"),
)

day20_runner = load_module("day20_runner", DAY20_RUNNER_PATH)

selected_conditions = [
    item for item in CONFIG["conditions"]
    if item["condition"] in ("rotation", "occlusion", "jpeg")
]
assert [
    (item["condition"], item["severity"])
    for item in selected_conditions
] == [
    (kind, severity)
    for kind in ("rotation", "occlusion", "jpeg")
    for severity in range(1, 5)
]

print("Day19 clean reference: VERIFIED")
print("Day20 runner SHA256:", sha_file(DAY20_RUNNER_PATH))
print("Conditions:", len(selected_conditions))
print("Image-condition inputs:", 19962 * 12)
print("Three-model sample evaluations:", 19962 * 12 * 3)
print("Day20 runner preparation: PASS")
print("No new inference performed")

Day19 clean reference: VERIFIED
Day20 runner SHA256: 26b1c71de8e10ac9f137f1bf7db5b236e98ff4f1909a50827385bbc97ddd95ef
Conditions: 12
Image-condition inputs: 239544
Three-model sample evaluations: 718632
Day20 runner preparation: PASS
No new inference performed


In [8]:
DAY20_REPORT = day20_runner.run_day20(**DAY20_CONTEXT)

Binding test JPEG bytes...
Official test inference starts: 12 conditions, 19962 images, 3 models

=== rotation_s1 ===
rotation_s1: 256/19962; saved; elapsed=0.4 min; remaining estimate=418.1 min
rotation_s1: 512/19962; saved; elapsed=0.9 min; remaining estimate=402.8 min
rotation_s1: 768/19962; saved; elapsed=1.2 min; remaining estimate=387.2 min
rotation_s1: 1024/19962; saved; elapsed=1.7 min; remaining estimate=385.2 min
rotation_s1: 1280/19962; saved; elapsed=2.1 min; remaining estimate=383.6 min
rotation_s1: 1536/19962; saved; elapsed=2.4 min; remaining estimate=374.8 min
rotation_s1: 1792/19962; saved; elapsed=2.8 min; remaining estimate=372.9 min
rotation_s1: 2048/19962; saved; elapsed=3.2 min; remaining estimate=372.9 min
rotation_s1: 2304/19962; saved; elapsed=3.6 min; remaining estimate=367.3 min
rotation_s1: 2560/19962; saved; elapsed=4.0 min; remaining estimate=367.0 min
rotation_s1: 2816/19962; saved; elapsed=4.3 min; remaining estimate=364.8 min
rotation_s1: 3072/19962; sa

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import json

PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
DAY20 = PROJECT / "runs/benchmark/day20"


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


report_path = DAY20 / "day20_benchmark_report.json"
assert report_path.is_file(), f"找不到最终报告：{report_path}"
report = json.loads(report_path.read_bytes())

assert report["day20_scope_complete"] is True
assert report["condition_count"] == 12
assert report["test_sample_count"] == 19962
assert report["unique_inputs"] == 239544
assert report["model_sample_evaluations"] == 718632
assert report["threshold_fitting_performed"] is False
assert report["runtime_errors"] == 0
assert len(report["shard_sha256"]) == 936

binding_path = DAY20 / "execution_binding.json"
assert sha_file(binding_path) == report["execution_binding_sha256"]
binding = json.loads(binding_path.read_bytes())

assert sha_file(
    DAY20 / "source/scripts/day20_evaluate_benchmark.py"
) == binding["runner_sha256"]

assert sha_file(
    DAY20 / "test_jpeg_sha256.json"
) == binding["test_jpeg_manifest_sha256"]

day19 = PROJECT / "runs/benchmark/day19"
assert sha_file(
    day19 / "execution_binding.json"
) == binding["day19_execution_binding_sha256"]
assert sha_file(
    day19 / "day19_benchmark_report.json"
) == binding["day19_report_sha256"]

expected_conditions = [
    (kind, severity)
    for kind in ("rotation", "occlusion", "jpeg")
    for severity in range(1, 5)
]
assert [
    (item["condition"]["condition"], item["condition"]["severity"])
    for item in report["conditions"]
] == expected_conditions

print("Condition       R18 F1    R50 F1     M3 F1    no_face")
for item in report["conditions"]:
    condition = item["condition"]
    label = f"{condition['condition']}_s{condition['severity']}"
    assert item["sample_count"] == 19962
    values = [
        item["models"][run]["frozen_thresholds"]["macro_f1"]
        for run in ("r18_seed0", "r50_seed0", "m3_seed0")
    ]
    print(
        f"{label:14s} "
        + "  ".join(f"{value:.6f}" for value in values)
        + f"  {item['detector']['no_face']}"
    )

print("\nDay20 final report and provenance: PASS")
print("Recorded prediction shards: 936")
print("尚未逐个重新校验预测 ZIP；下一步归档时执行。")

Mounted at /content/drive
Condition       R18 F1    R50 F1     M3 F1    no_face
rotation_s1    0.743728  0.737617  0.741183  22
rotation_s2    0.718881  0.714504  0.721719  26
rotation_s3    0.682277  0.676971  0.698202  36
rotation_s4    0.641866  0.641634  0.665834  49
occlusion_s1   0.723695  0.719730  0.709241  158
occlusion_s2   0.584136  0.590099  0.561994  1014
occlusion_s3   0.516736  0.517627  0.492081  2018
occlusion_s4   0.441348  0.422453  0.409276  4154
jpeg_s1        0.754178  0.752744  0.754459  19
jpeg_s2        0.749693  0.747136  0.749247  21
jpeg_s3        0.725181  0.724475  0.729007  23
jpeg_s4        0.613221  0.612502  0.591405  30

Day20 final report and provenance: PASS
Recorded prediction shards: 936
尚未逐个重新校验预测 ZIP；下一步归档时执行。


In [2]:
from google.colab import files
import copy
import zipfile

# 1. 确认每个条件的批次范围完整，无缺失或额外 ZIP。
expected_shards = set()
for item in report["conditions"]:
    condition = item["condition"]
    label = f"{condition['condition']}_s{condition['severity']}"
    for start in range(0, 19962, 256):
        stop = min(start + 256, 19962)
        expected_shards.add(
            f"shards/{label}/{start:06d}_{stop:06d}.zip"
        )

assert set(report["shard_sha256"]) == expected_shards
actual_shards = {
    path.relative_to(DAY20).as_posix()
    for path in (DAY20 / "shards").rglob("*.zip")
}
assert actual_shards == expected_shards

# ZIP 整体 SHA 与最终报告一致，即确认保存后的内容未变化。
for index, relative in enumerate(sorted(expected_shards), 1):
    assert sha_file(DAY20 / relative) == report["shard_sha256"][relative], (
        f"预测批次校验失败：{relative}"
    )
    if index % 100 == 0 or index == 936:
        print(f"Verified prediction shards: {index}/936", flush=True)

# 2. 收集报告，并检查各条件报告与总报告一致。
payloads = {}
for name in (
    "day20_benchmark_report.json",
    "execution_binding.json",
    "test_jpeg_sha256.json",
):
    payloads[name] = (DAY20 / name).read_bytes()

for item in report["conditions"]:
    condition = item["condition"]
    label = f"{condition['condition']}_s{condition['severity']}"
    relative = f"condition_reports/{label}.json"
    data = (DAY20 / relative).read_bytes()

    # 单条件报告写入时尚未附加相对 clean 的下降值。
    expected = copy.deepcopy(item)
    for metrics in expected["models"].values():
        metrics.pop("frozen_thresholds_degradation_pp", None)
        metrics.pop("fixed_0_5_degradation_pp", None)

    assert json.loads(data) == expected, relative
    payloads[relative] = data

runner_path = DAY20 / "source/scripts/day20_evaluate_benchmark.py"
assert sha_file(runner_path) == binding["runner_sha256"]
payloads["scripts/day20_evaluate_benchmark.py"] = runner_path.read_bytes()

# 保留之前运行错误记录（如有），供复盘。
for relative, expected_sha in report["prior_runtime_error_attempts"].items():
    path = DAY20 / relative
    assert sha_file(path) == expected_sha, relative
    payloads[relative] = path.read_bytes()

manifest = {
    "scope": "Day20 rotation, occlusion and JPEG; 12 conditions",
    "test_sample_count": 19962,
    "prediction_shards_verified": 936,
    "execution_binding_sha256": report["execution_binding_sha256"],
    "threshold_fitting_performed": False,
    "prediction_shards_location": "runs/benchmark/day20/shards/",
    "files": {
        name: hashlib.sha256(data).hexdigest()
        for name, data in sorted(payloads.items())
    },
}
payloads["archive_manifest.json"] = (
    json.dumps(manifest, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

# 3. 导出到当前 Colab；读取并验证打包内容后下载。
archive_path = Path("/content/day20_results_archive.zip")
with zipfile.ZipFile(
    archive_path, "w", zipfile.ZIP_DEFLATED
) as archive:
    for name, data in sorted(payloads.items()):
        archive.writestr(name, data)

with zipfile.ZipFile(archive_path) as archive:
    assert len(archive.namelist()) == len(payloads)
    assert set(archive.namelist()) == set(payloads)
    for name, data in payloads.items():
        assert archive.read(name) == data, name

print("Day20 shard verification and archive: PASS")
print("Archive SHA256:", sha_file(archive_path))
files.download(str(archive_path))

Verified prediction shards: 100/936
Verified prediction shards: 200/936
Verified prediction shards: 300/936
Verified prediction shards: 400/936
Verified prediction shards: 500/936
Verified prediction shards: 600/936
Verified prediction shards: 700/936
Verified prediction shards: 800/936
Verified prediction shards: 900/936
Verified prediction shards: 936/936
Day20 shard verification and archive: PASS
Archive SHA256: 03e13995d72caf77a1576796564a9687139419c9b300ecfe3ff81472486ce313


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>